### 1. Delta 編碼與解碼
這段程式碼負責將資料轉為差值。請注意 % 256 的運用，這能確保所有資料在相減後依然完美的保持在 0 ~ 255 的無號位元組（Unsigned Byte）範圍內。

In [ ]:
def delta_encode(data: bytes) -> bytearray:
    """將輸入的 bytes 進行 Delta (差值) 編碼"""
    if not data:
        return bytearray()
    
    output = bytearray(len(data))
    output[0] = data[0]  # 第一個元素保持原樣
    
    # 每個 byte 減去前一個 byte，並透過 % 256 限制在 0-255 之間
    for i in range(1, len(data)):
        output[i] = (data[i] - data[i-1]) % 256
    return output

def delta_decode(data: bytearray) -> bytes:
    """將 Delta 編碼後的資料還原"""
    if not data:
        return b""
    
    output = bytearray(len(data))
    output[0] = data[0]
    
    # 每個編碼後的 byte 加上前一個還原後的 byte
    for i in range(1, len(data)):
        output[i] = (data[i] + output[i-1]) % 256
    return bytes(output)

# --- 快速驗證 Delta ---
test_text = b"aaaaabbbbbccccc12345"
encoded_delta = delta_encode(test_text)
decoded_delta = delta_decode(encoded_delta)

print("【Delta 測試】")
print(f"原始資料: {test_text}")
print(f"Delta 後 : {list(encoded_delta)}") # 你會看到大量重複的 0 或固定值
print(f"還原成功: {decoded_delta == test_text}")

### 2. LZ77 壓縮與解壓縮（含雜湊優化）

透過滑動視窗尋找歷史重複字串，並轉化為 `(is_match, literal/distance, length)` 的 Token 串流。

- **參數防禦限制**：
    
    - **滑動視窗 (Window Size)**：嚴格鎖定在 `3000` 位元組以內（絕對小於 12-bit 的 $4095$），防禦邊界溢出。
        
    - **最大匹配長度 (Max Match Length)**：鎖定在 `255` 以內（嚴格符合 8-bit 上限）。
        
- **效能優化**：利用三字元組雜湊表（`pos_hash`）快速索引歷史位置，將動態匹配時間大幅縮短。

In [ ]:
def lz77_compress(data: bytes, window_size: int = 3000) -> list:
    """
    將視窗嚴格鎖定在 3000 (絕對小於 12-bit 的 4095)，防禦邊界溢出
    """
    tokens = []
    cursor = 0
    data_len = len(data)
    max_match_len = 255  # 嚴格鎖定在 8-bit 的 255 以內
    
    pos_hash = {}
    
    while cursor < data_len:
        match_dist = 0
        match_len = 0
        
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            p = pos_hash.get(triple, -1)
            
            # 確保歷史位置在合法的 3000 位元組視窗內，且嚴格小於當前 cursor
            if p != -1 and (cursor - p <= window_size) and (p < cursor):
                curr_match_len = 0
                while (cursor + curr_match_len < data_len and 
                       data[p + curr_match_len] == data[cursor + curr_match_len] and 
                       curr_match_len < max_match_len):
                    curr_match_len += 1
                    
                if curr_match_len >= 3:
                    match_len = curr_match_len
                    match_dist = cursor - p

        if match_len >= 3:
            # 安全防禦：再次確保數值絕對沒有超出 Bit 承載上限
            if 0 < match_dist <= 4095 and 3 <= match_len <= 255:
                tokens.append((True, match_dist, match_len))
                if cursor + 3 <= data_len:
                    triple = (data[cursor], data[cursor+1], data[cursor+2])
                    pos_hash[triple] = cursor
                cursor += match_len
                continue
                
        # 沒找到匹配或是數值異常，一律退回字面值
        tokens.append((False, data[cursor], 0))
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            pos_hash[triple] = cursor
        cursor += 1
            
    return tokens

def lz77_decompress(tokens: list) -> bytes:
    output = bytearray()
    for is_match, val, length in tokens:
        if not is_match:
            output.append(val)
        else:
            distance = val
            start_pos = len(output) - distance
            for i in range(length):
                output.append(output[start_pos + i])
    return bytes(output)

### 3. 進階核心：動態智慧決策與雙模式分流

系統在打包前會先進入「模擬估算模式」，精準計算兩種模式所需的總位元組（包含 Header 與 Bit 流開銷），並自動採取空間最低的最優解：

- **【模式 0】單樹規範化哈夫曼模式**
    
    - **適合對象**：中小型純文字檔。
        
    - **機制**：將字面值（0~255）與長度/距離（257）合在一棵樹，重複字串的距離與長度固定佔用 12+8 bits。能有效避免小型檔案因多個長度表而導致 Header 過大。
        
- **【模式 1】二級規範化哈夫曼雙樹模式**
    
    - **適合對象**：大型點陣圖或影像檔。
        
    - **機制**：拆分為「樹 A（字面值/長度表）」與「樹 B（距離表）」。連字典距離（0~4095）都改用變長度哈夫曼編碼動態壓縮，並引入**邊界動態裁切**技術捨棄無效的 `0` 欄位。

In [ ]:
import struct
from collections import Counter
import heapq

# --- 保持穩定的硬核 Bit 工具 ---
class BitWriter:
    def __init__(self):
        self.bytes_data = bytearray()
        self.buffer = 0
        self.bit_count = 0
    def write_bits(self, value: int, num_bits: int):
        value = value & ((1 << num_bits) - 1)
        for i in range(num_bits - 1, -1, -1):
            bit = (value >> i) & 1
            self.buffer = (self.buffer << 1) | bit
            self.bit_count += 1
            if self.bit_count == 8:
                self.bytes_data.append(self.buffer)
                self.buffer = 0
                self.bit_count = 0
    def flush(self):
        if self.bit_count > 0:
            self.buffer = self.buffer << (8 - self.bit_count)
            self.bytes_data.append(self.buffer)
            self.buffer = 0
            self.bit_count = 0
        return self.bytes_data

class BitReader:
    def __init__(self, data: bytes):
        self.data = data
        self.byte_idx = 0
        self.bit_idx = 7
    def read_bit(self) -> int:
        if self.byte_idx >= len(self.data):
            return 0
        bit = (self.data[self.byte_idx] >> self.bit_idx) & 1
        self.bit_idx -= 1
        if self.bit_idx < 0:
            self.bit_idx = 7
            self.byte_idx += 1
        return bit

# --- 規範化哈夫曼生成器 ---
def generate_canonical_codes(code_lengths: dict) -> dict:
    if not code_lengths: return {}
    max_len = max(code_lengths.values())
    bl_count = Counter(code_lengths.values())
    
    next_code = {}
    code = 0
    bl_count[0] = 0
    for bits in range(1, max_len + 1):
        code = (code + bl_count[bits - 1]) << 1
        next_code[bits] = code
        
    canonical_codes = {}
    for sym in sorted(code_lengths.keys()):
        length = code_lengths[sym]
        if length > 0:
            code_val = next_code[length]
            canonical_codes[sym] = format(code_val, f'0{length}b')
            next_code[length] += 1
    return canonical_codes

def get_huffman_lengths(frequencies: dict, max_symbols: int) -> dict:
    """計算指定符號上限的哈夫曼編碼長度"""
    if len(frequencies) == 0: 
        return {i: 0 for i in range(max_symbols)}
    if len(frequencies) == 1:
        sym = list(frequencies.keys())[0]
        return {i: (1 if i == sym else 0) for i in range(max_symbols)}
        
    heap = [[wt, sym, [sym]] for sym, wt in frequencies.items()]
    heapq.heapify(heap)
    
    lengths = {sym: 0 for sym in frequencies}
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for sym in lo[2]: lengths[sym] += 1
        for sym in hi[2]: lengths[sym] += 1
        heapq.heappush(heap, [lo[0] + hi[0], min(lo[1], hi[1]), lo[2] + hi[2]])
        
    # 補齊未出現的符號長度為 0
    full_lengths = {i: 0 for i in range(max_symbols)}
    for sym, l in lengths.items():
        full_lengths[sym] = l
    return full_lengths

### 4. 智慧型動態檔案標頭 (Header Box) 格式

壓縮檔開頭寫入 1 字节的 `chosen_mode`，使還原端能自動解流：

|**欄位名稱**|**佔用空間**|**說明**|
|---|---|---|
|**Magic Number**|2 Bytes|固定為 `b"MY"`，用於辨識檔案格式。|
|**Original Size**|4 Bytes|無號整數（Little-endian），記錄原始檔案大小。|
|**Chosen Mode**|1 Byte|**0 代表單樹模式；1 代表雙樹模式**。|

- **若為模式 0**：接續寫入 258 位元組的固定長度表。
    
- **若為模式 1**：接續寫入樹 A 最大有效位置（2B）+ 樹 A 長度數據 + 樹 B 最大有效位置（2B）+ 樹 B 長度數據。

In [ ]:
import struct
from collections import Counter
import heapq

# ==================== 1. 底層硬核 Bit 工具 ====================
class BitWriter:
    def __init__(self):
        self.bytes_data = bytearray()
        self.buffer = 0
        self.bit_count = 0
    def write_bits(self, value: int, num_bits: int):
        value = value & ((1 << num_bits) - 1)
        for i in range(num_bits - 1, -1, -1):
            bit = (value >> i) & 1
            self.buffer = (self.buffer << 1) | bit
            self.bit_count += 1
            if self.bit_count == 8:
                self.bytes_data.append(self.buffer)
                self.buffer = 0
                self.bit_count = 0
    def flush(self):
        if self.bit_count > 0:
            self.buffer = self.buffer << (8 - self.bit_count)
            self.bytes_data.append(self.buffer)
            self.buffer = 0
            self.bit_count = 0
        return self.bytes_data

class BitReader:
    def __init__(self, data: bytes):
        self.data = data
        self.byte_idx = 0
        self.bit_idx = 7
    def read_bit(self) -> int:
        if self.byte_idx >= len(self.data):
            return 0
        bit = (self.data[self.byte_idx] >> self.bit_idx) & 1
        self.bit_idx -= 1
        if self.bit_idx < 0:
            self.bit_idx = 7
            self.byte_idx += 1
        return bit
    def read_bits(self, num_bits: int) -> int:
        value = 0
        for _ in range(num_bits):
            value = (value << 1) | self.read_bit()
        return value

# ==================== 2. 規範化哈夫曼演算法核心 ====================
def generate_canonical_codes(code_lengths: dict) -> dict:
    if not code_lengths: return {}
    max_len = max(code_lengths.values())
    bl_count = Counter(code_lengths.values())
    
    next_code = {}
    code = 0
    bl_count[0] = 0
    for bits in range(1, max_len + 1):
        code = (code + bl_count[bits - 1]) << 1
        next_code[bits] = code
        
    canonical_codes = {}
    for sym in sorted(code_lengths.keys()):
        length = code_lengths[sym]
        if length > 0:
            code_val = next_code[length]
            canonical_codes[sym] = format(code_val, f'0{length}b')
            next_code[length] += 1
    return canonical_codes

def get_huffman_lengths(frequencies: dict, max_symbols: int) -> dict:
    if len(frequencies) == 0: 
        return {i: 0 for i in range(max_symbols)}
    # differ method $
    if len(frequencies) == 1:
        sym = list(frequencies.keys())[0]
        return {i: (1 if i == sym else 0) for i in range(max_symbols)}
        
    heap = [[wt, sym, [sym]] for sym, wt in frequencies.items()]
    heapq.heapify(heap)
    
    lengths = {sym: 0 for sym in frequencies}
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for sym in lo[2]: lengths[sym] += 1
        for sym in hi[2]: lengths[sym] += 1
        heapq.heappush(heap, [lo[0] + hi[0], min(lo[1], hi[1]), lo[2] + hi[2]])
        
    full_lengths = {i: 0 for i in range(max_symbols)}
    for sym, l in lengths.items():
        full_lengths[sym] = l
    return full_lengths

# ==================== 3. 終極動態自動分流管道 ====================

def my_custom_compress(original_data: bytes) -> bytes:
    if not original_data:
        return b""
    
    # 共同前置：Delta + LZ77
    delta_data = delta_encode(original_data)
    lz77_tokens = lz77_compress(delta_data)
    
    # ------------------ 模擬估算模式 0 (單樹模式) ------------------
    m0_frequencies = Counter()
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m0_frequencies[val] += 1
        else:
            m0_frequencies[257] += 1
    m0_frequencies[256] += 1 # 結束符
    
    m0_lengths = get_huffman_lengths(m0_frequencies, 258)
    # 計算模式 0 位元組總數 = 固定 Header 6B + 長度表 258B + Bit流預估
    m0_bit_payload = 0
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m0_bit_payload += m0_lengths[val]
        else:
            m0_bit_payload += m0_lengths[257] + 12 + 8 # 加上固定的 12+8 bits
    m0_bit_payload += m0_lengths[256]
    m0_total_estimate = 6 + 1 + 258 + ((m0_bit_payload + 7) // 8) # 6B(Base)+1B(Mode)+258B(Tree)
    
    # ------------------ 模擬估算模式 1 (雙樹模式) ------------------
    m1_lit_len_freqs = Counter()
    m1_dist_freqs = Counter()
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m1_lit_len_freqs[val] += 1
        else:
            len_sym = 257 + (length - 3)
            m1_lit_len_freqs[len_sym] += 1
            m1_dist_freqs[val] += 1
    m1_lit_len_freqs[256] += 1
    
    m1_lit_len_lengths = get_huffman_lengths(m1_lit_len_freqs, 512)
    m1_dist_lengths = get_huffman_lengths(m1_dist_freqs, 4096)
    
    max_lit_len_used = max([i for i, l in m1_lit_len_lengths.items() if l > 0] + [257])
    max_dist_used = max([i for i, l in m1_dist_lengths.items() if l > 0] + [0])
    
    m1_bit_payload = 0
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m1_bit_payload += m1_lit_len_lengths[val]
        else:
            len_sym = 257 + (length - 3)
            m1_bit_payload += m1_lit_len_lengths[len_sym] + m1_dist_lengths[val]
    m1_bit_payload += m1_lit_len_lengths[256]
    # Header: Base(6B) + Mode(1B) + TreeA_Len(2B) + TreeA_Data + TreeB_Len(2B) + TreeB_Data
    m1_total_estimate = 6 + 1 + 2 + (max_lit_len_used + 1) + 2 + (max_dist_used + 1) + ((m1_bit_payload + 7) // 8)

    # ------------------ 🌟 動態智慧決策 ------------------
    chosen_mode = 0 if m0_total_estimate <= m1_total_estimate else 1
    
    # ------------------ 真正開始進行打包 ------------------
    header = bytearray()
    header += struct.pack("<2sI", b"MY", len(original_data)) # Base Header
    header.append(chosen_mode) # 寫入模式 Flag (1 Byte)
    
    writer = BitWriter()
    
    if chosen_mode == 0:
        # 【實作單樹模式】
        # 1. 寫入長度表
        for sym in range(258):
            header.append(m0_lengths[sym])
        # 2. 寫入 Bit 流
        huff_table = generate_canonical_codes({k: v for k, v in m0_lengths.items() if v > 0})
        for is_match, val, length in lz77_tokens:
            if not is_match:
                bit_str = huff_table[val]
                writer.write_bits(int(bit_str, 2), len(bit_str))
            else:
                bit_str = huff_table[257]
                writer.write_bits(int(bit_str, 2), len(bit_str))
                writer.write_bits(val, 12)
                writer.write_bits(length, 8)
        end_str = huff_table[256]
        writer.write_bits(int(end_str, 2), len(end_str))
        
    else:
        # 【實作雙樹模式】
        # 1. 寫入樹 A
        header += struct.pack("<H", max_lit_len_used)
        for i in range(max_lit_len_used + 1):
            header.append(m1_lit_len_lengths[i])
        # 2. 寫入樹 B
        header += struct.pack("<H", max_dist_used)
        for i in range(max_dist_used + 1):
            header.append(m1_dist_lengths[i])
        # 3. 寫入 Bit 流
        lit_len_table = generate_canonical_codes({k: v for k, v in m1_lit_len_lengths.items() if v > 0})
        dist_table = generate_canonical_codes({k: v for k, v in m1_dist_lengths.items() if v > 0})
        
        for is_match, val, length in lz77_tokens:
            if not is_match:
                bit_str = lit_len_table[val]
                writer.write_bits(int(bit_str, 2), len(bit_str))
            else:
                len_sym = 257 + (length - 3)
                bit_str_len = lit_len_table[len_sym]
                writer.write_bits(int(bit_str_len, 2), len(bit_str_len))
                bit_str_dist = dist_table[val]
                writer.write_bits(int(bit_str_dist, 2), len(bit_str_dist))
        end_str = lit_len_table[256]
        writer.write_bits(int(end_str, 2), len(end_str))
        
    return bytes(header) + writer.flush()


def my_custom_decompress(compressed_bytes: bytes) -> bytes:
    if not compressed_bytes:
        return b""
        
    magic, orig_size = struct.unpack("<2sI", compressed_bytes[:6])
    if magic != b"MY":
        raise ValueError("不合法的壓縮檔案格式！")
        
    # 讀取模式 Flag
    chosen_mode = compressed_bytes[6]
    idx = 7
    
    output = bytearray()
    
    if chosen_mode == 0:
        # 【解碼單樹模式】
        code_lengths = {}
        for sym in range(258):
            length = compressed_bytes[idx]
            if length > 0: code_lengths[sym] = length
            idx += 1
            
        reverse_table = {bits: sym for sym, bits in generate_canonical_codes(code_lengths).items()}
        bit_data = compressed_bytes[idx:]
        reader = BitReader(bit_data)
        
        curr_bits = ""
        while True:
            curr_bits += str(reader.read_bit())
            if curr_bits in reverse_table:
                sym = reverse_table[curr_bits]
                curr_bits = ""
                if sym == 256: break
                elif sym <= 255: output.append(sym)
                elif sym == 257:
                    distance = reader.read_bits(12)
                    length = reader.read_bits(8)
                    if distance > len(output): distance = len(output)
                    if distance == 0:
                        for _ in range(length): output.append(0)
                        continue
                    start_pos = len(output) - distance
                    for _ in range(length):
                        output.append(output[start_pos])
                        start_pos += 1
                        
    else:
        # 【解碼雙樹模式】
        max_lit_len_used = struct.unpack("<H", compressed_bytes[idx:idx+2])[0]
        idx += 2
        lit_len_lengths = {i: 0 for i in range(512)}
        for i in range(max_lit_len_used + 1):
            lit_len_lengths[i] = compressed_bytes[idx]
            idx += 1
        reverse_lit_len_table = {bits: sym for sym, bits in generate_canonical_codes(lit_len_lengths).items()}
        
        max_dist_used = struct.unpack("<H", compressed_bytes[idx:idx+2])[0]
        idx += 2
        dist_lengths = {i: 0 for i in range(4096)}
        for i in range(max_dist_used + 1):
            dist_lengths[i] = compressed_bytes[idx]
            idx += 1
        reverse_dist_table = {bits: sym for sym, bits in generate_canonical_codes(dist_lengths).items()}
        
        bit_data = compressed_bytes[idx:]
        reader = BitReader(bit_data)
        
        curr_bits = ""
        while True:
            curr_bits += str(reader.read_bit())
            if curr_bits in reverse_lit_len_table:
                sym = reverse_lit_len_table[curr_bits]
                curr_bits = ""
                if sym == 256: break
                elif sym <= 255: output.append(sym)
                elif sym >= 257:
                    length = (sym - 257) + 3
                    dist_bits = ""
                    while True:
                        dist_bits += str(reader.read_bit())
                        if dist_bits in reverse_dist_table:
                            distance = reverse_dist_table[dist_bits]
                            break
                    if distance > len(output): distance = len(output)
                    if distance == 0:
                        for _ in range(length): output.append(0)
                        continue
                    start_pos = len(output) - distance
                    for _ in range(length):
                        output.append(output[start_pos])
                        start_pos += 1
                        
    return delta_decode(output)

### 5. 自動化基準測試與數據結論

在 v4 的智慧決策管線下，系統在各種類型檔案均成功奪回「最佳化平衡」：

```
【空間節省率 (%) 對比表】
檔案名稱       | v3 版本   | v4 智慧分流版 | 狀態效益說明
------------------------------------------------------------------
test1.txt     | -731.43% |  -720.00%   | 🛡️ 止血！小檔自動避開雙樹 Header 開銷
test2.txt     |  -49.89% |  +24.45%    | 🚀 逆轉！純文字自動切回模式 0，空間由負轉正
test3.txt     |  -16.88% |  +27.72%    | 🚀 逆轉！大純文字檔完美恢復 1.38x 壓縮水準
Lenna.bmp     |  +30.51% |  +30.51%    | 💎 保留！影像檔完美維持雙樹的 30% 極致壓縮
Cameraman.bmp |  +26.97% |  +26.97%    | 💎 保留！影像檔完美維持雙樹的 26% 極致壓縮
```

- **結論**：v4 完美解決了先前 v3 對純文字檔「負優化」的嚴重缺陷。系統在面對 **BMP 影像時能維持 30% 的極致壓縮（使用雙樹）**，面對 **TXT 文字時則能自動切換模式，穩拿 24%~27% 的空間節省**，全數通過（PASS）還原驗證！

In [ ]:
import time
import os
import pandas as pd

def run_benchmark_pandas(file_list):
    """
    自動測試檔案列表，並使用 Pandas DataFrame 輸出精美報表
    """
    results = []
    
    for file_name in file_list:
        if not os.path.exists(file_name):
            print(f"⚠️ 找不到檔案: {file_name}，已跳過。")
            continue
            
        # 1. 讀取原始檔案
        with open(file_name, "rb") as f:
            original_data = f.read()
            
        orig_size = len(original_data)
        if orig_size == 0:
            continue
            
        # 2. 測試壓縮時間
        t0 = time.perf_counter()
        compressed_data = my_custom_compress(original_data)
        t1 = time.perf_counter()
        comp_time_ms = (t1 - t0) * 1000
        
        comp_size = len(compressed_data)
        
        # 3. 測試解壓時間
        t2 = time.perf_counter()
        decompressed_data = my_custom_decompress(compressed_data)
        t3 = time.perf_counter()
        decomp_time_ms = (t3 - t2) * 1000
        
        # 4. 計算指標
        comp_ratio = orig_size / comp_size if comp_size > 0 else 0
        space_saving = ((orig_size - comp_size) / orig_size) * 100
        is_valid = "成功 (PASS)" if decompressed_data == original_data else "失敗 (FAIL)"
        
        # 5. 紀錄數據
        results.append({
            "檔案名稱": file_name,
            "原始大小 (Bytes)": orig_size,
            "壓縮大小 (Bytes)": comp_size,
            "壓縮時間 (ms)": round(comp_time_ms, 2),
            "解壓時間 (ms)": round(decomp_time_ms, 2),
            "壓縮率 (倍數)": f"{comp_ratio:.2f}x",
            "空間節省 (%)": f"{space_saving:.2f}%",
            "解壓驗證": is_valid
        })
    
    # 6. 轉換為 DataFrame 並格式化顯示
    df = pd.DataFrame(results)
    
    # 對千分位進行格式化處理，讓報告更好看
    df["原始大小 (Bytes)"] = df["原始大小 (Bytes)"].map("{:,}".format)
    df["壓縮大小 (Bytes)"] = df["壓縮大小 (Bytes)"].map("{:,}".format)
    
    return df

# --- 執行測試 ---
target_files = [
    "test1.txt",
    "test2.txt",
    "test3.txt",
    "Lenna.bmp",
    "Cameraman.bmp"
]

# 執行並顯示 DataFrame
benchmark_df = run_benchmark_pandas(target_files)
benchmark_df